# La mémoire des agents LLM — notebook d'accompagnement

Atelier interne Data Engineering / Data Science — Session 2 (pratique).

Support des slides : `docs/slides/agent-memory-part1-theory.html` et
`docs/slides/agent-memory-part2-practice.html`.

**Plan :**
1. Similarité cosinus « from scratch » (numpy)
2. Score de récupération façon *Generative Agents* (recency + importance + relevance)
3. LangGraph — mémoire courte terme (checkpointer, threads, time travel)
4. LangGraph — mémoire long terme (`Store`, namespaces, recherche sémantique, `langmem`)
5. DeepAgents — sous-agents, filesystem virtuel, persistance cross-session

**Important — exécution avec ou sans clé API :**
Les sections 1 et 2 sont 100% locales (numpy uniquement, pas de LLM). Les sections
3 à 5 utilisent un vrai modèle Anthropic si la variable d'environnement
`ANTHROPIC_API_KEY` est définie, et basculent automatiquement sur un modèle
factice (déterministe, hors-ligne) sinon — pour que tout le monde dans l'équipe
puisse exécuter le notebook de bout en bout avant la session, avec ou sans accès API.


In [ ]:
# Installation (à exécuter une seule fois, décommenter au besoin)
# %pip install -q "langgraph>=1.2,<2.0" "langchain>=1.4,<2.0" "langmem>=0.0.30" \
#     "deepagents>=0.7,<0.8" "langchain-anthropic" numpy

import os
import math
import uuid
import time
from datetime import datetime, timedelta

import numpy as np

ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY")
HAS_ANTHROPIC = bool(ANTHROPIC_API_KEY)
print("Clé ANTHROPIC_API_KEY détectée :", HAS_ANTHROPIC)
if not HAS_ANTHROPIC:
    print("-> Les sections 3 à 5 utiliseront un modèle factice hors-ligne pour la démo.")


## 1. Similarité cosinus, from scratch

La quasi-totalité des systèmes de « mémoire vectorielle » repose sur une seule
formule :

$$\cos(u, v) = \frac{u \cdot v}{\lVert u \rVert \, \lVert v \rVert}$$

Pour ne dépendre d'aucune clé API à ce stade, on utilise un **embedding jouet**
(hachage de sacs de mots) plutôt qu'un vrai modèle d'embedding — l'idée reste
strictement la même avec de vrais embeddings (OpenAI, Voyage, Cohere, ...).


In [ ]:
def toy_embed(text: str, dims: int = 256) -> np.ndarray:
    """Embedding jouet et déterministe (hashing de mots) — pour la démo hors-ligne.
    Remplacer par un vrai modèle d'embedding en production."""
    vec = np.zeros(dims, dtype=float)
    for word in text.lower().split():
        idx = hash(word) % dims
        vec[idx] += 1.0
    norm = np.linalg.norm(vec)
    return vec / norm if norm > 0 else vec


def cosine_similarity(u: np.ndarray, v: np.ndarray) -> float:
    denom = np.linalg.norm(u) * np.linalg.norm(v)
    return float(np.dot(u, v) / denom) if denom > 0 else 0.0


phrases = [
    "le pipeline airflow a échoué cette nuit",
    "le dag airflow est tombé en erreur hier soir",
    "le client préfère les réponses concises et le code d'abord",
    "recette de tarte aux pommes pour ce week-end",
]

query = "pourquoi le job airflow a-t-il planté ?"
q_vec = toy_embed(query)

print(f"Requête : {query!r}\n")
for p in phrases:
    sim = cosine_similarity(q_vec, toy_embed(p))
    print(f"  cos = {sim:0.3f}   {p!r}")


On voit les deux phrases sur le pipeline Airflow ressortir nettement au-dessus
des deux autres — c'est exactement le mécanisme (avec de vrais embeddings à la
place du hachage jouet) qui fait tourner la recherche sémantique du `Store`
LangGraph en section 4.


## 2. Score de récupération façon *Generative Agents*

Park et al. (2023) combinent trois signaux pour décider quel souvenir rappeler :

$$\text{score} = \alpha_r \cdot \text{recency} + \alpha_i \cdot \text{importance} + \alpha_v \cdot \text{relevance}$$

- **recency** : décroissance exponentielle, $\gamma^{\Delta t}$ avec $\gamma = 0.995$
  et $\Delta t$ en « heures » depuis le dernier accès.
- **importance** : notée 1–10 à l'écriture (ici on la fixe à la main, comme un tag).
- **relevance** : similarité cosinus entre le souvenir et la requête actuelle.

Chaque terme est normalisé min-max dans [0, 1] avant la somme (poids égaux ici,
$\alpha_r = \alpha_i = \alpha_v = 1$).


In [ ]:
GAMMA = 0.995

memories = [
    {"text": "Incident Airflow : DAG `daily_ingest` en échec, retry manuel nécessaire",
     "importance": 7, "hours_ago": 2},
    {"text": "L'utilisateur a dit préférer des réponses concises, code d'abord",
     "importance": 6, "hours_ago": 240},
    {"text": "Recette de tarte aux pommes partagée dans le canal #random",
     "importance": 1, "hours_ago": 20},
    {"text": "Le DAG `daily_ingest` avait déjà échoué le mois dernier pour la même raison",
     "importance": 5, "hours_ago": 720},
]

query = "le job airflow a encore planté, que faire ?"
q_vec = toy_embed(query)


def min_max(values):
    lo, hi = min(values), max(values)
    if hi == lo:
        return [1.0 for _ in values]
    return [(v - lo) / (hi - lo) for v in values]


recencies = [GAMMA ** m["hours_ago"] for m in memories]
importances = [m["importance"] for m in memories]
relevances = [cosine_similarity(q_vec, toy_embed(m["text"])) for m in memories]

r_n, i_n, v_n = min_max(recencies), min_max(importances), min_max(relevances)
scores = [r + i + v for r, i, v in zip(r_n, i_n, v_n)]

ranked = sorted(zip(memories, scores, recencies, importances, relevances),
                 key=lambda x: x[1], reverse=True)

print(f"{'score':>6}  {'recency':>8}  {'import.':>8}  {'relevance':>9}   souvenir")
for m, s, rec, imp, rel in ranked:
    print(f"{s:6.2f}  {rec:8.3f}  {imp:8d}  {rel:9.3f}   {m['text']}")


Remarquez : le souvenir « déjà échoué le mois dernier » (relevance élevée mais
recency très faible) et « préfère des réponses concises » (recency moyenne,
relevance faible mais importance notable) se disputent la 2ᵉ/3ᵉ place — c'est
exactement le genre d'arbitrage que ce score rend explicite plutôt qu'implicite.
Changez `GAMMA`, les `hours_ago` ou les poids $\alpha$ pour voir l'effet.


## 3. LangGraph — mémoire courte terme (checkpointer)

On construit un graphe minimal à un nœud, et on observe ce que le
`checkpointer` + `thread_id` apportent : persistance par conversation, et
isolation stricte entre threads.


In [ ]:
from langgraph.graph import StateGraph, MessagesState, START
from langgraph.checkpoint.memory import InMemorySaver  # nom canonique ; "MemorySaver" reste un alias
from langchain_core.messages import AIMessage, HumanMessage


class _EchoLLM:
    """Modèle factice, déterministe, pour exécuter la démo sans clé API."""

    def invoke(self, messages):
        last_user = next((m.content for m in reversed(messages)
                           if isinstance(m, HumanMessage)), "")
        seen_names = [w for m in messages if isinstance(m, HumanMessage)
                      for w in m.content.split() if w.istitle()]
        if "prénom" in last_user.lower() or "nom" in last_user.lower():
            reply = f"[mock-llm] Tu m'as dit t'appeler : {seen_names[-1] if seen_names else '???'}"
        else:
            reply = f"[mock-llm] j'ai bien reçu : {last_user!r}"
        return AIMessage(content=reply)


if HAS_ANTHROPIC:
    from langchain_anthropic import ChatAnthropic
    llm = ChatAnthropic(model="claude-haiku-4-5-20251001", temperature=0)
else:
    llm = _EchoLLM()


def call_model(state: MessagesState):
    response = llm.invoke(state["messages"])
    return {"messages": [response]}


graph = StateGraph(MessagesState)
graph.add_node("call_model", call_model)
graph.add_edge(START, "call_model")

checkpointer = InMemorySaver()  # dev uniquement — PostgresSaver en production
app = graph.compile(checkpointer=checkpointer)


In [ ]:
config_a = {"configurable": {"thread_id": "user-42-session-A"}}

app.invoke({"messages": [HumanMessage(content="Bonjour, je suis Alex")]}, config=config_a)
result = app.invoke({"messages": [HumanMessage(content="Quel est mon prénom ?")]}, config=config_a)
print("Thread A ->", result["messages"][-1].content)

config_b = {"configurable": {"thread_id": "user-42-session-B"}}
result_b = app.invoke({"messages": [HumanMessage(content="Quel est mon prénom ?")]}, config=config_b)
print("Thread B (nouveau thread_id) ->", result_b["messages"][-1].content)


Le thread A « se souvient » du prénom (même `thread_id`, checkpoints
enchaînés) ; le thread B ne sait rien — il n'a jamais vu de checkpoint pour ce
`thread_id`. C'est toute la mécanique de la mémoire courte terme LangGraph.


In [ ]:
# Time travel : lister l'historique des checkpoints du thread A, et revenir en arrière
history = list(app.get_state_history(config_a))
print(f"{len(history)} checkpoints enregistrés pour le thread A")
for snap in history:
    n_msgs = len(snap.values.get("messages", []))
    print(f"  checkpoint {snap.config['configurable']['checkpoint_id'][:8]}...  "
          f"({n_msgs} messages, next={snap.next})")

# Reprendre exactement depuis un checkpoint antérieur précis :
if len(history) >= 2:
    earlier_checkpoint_id = history[-1].config["configurable"]["checkpoint_id"]
    earlier_config = {**config_a, "configurable": {**config_a["configurable"],
                                                     "checkpoint_id": earlier_checkpoint_id}}
    earlier_state = app.get_state(earlier_config)
    print("\nÉtat au premier checkpoint :", [m.content for m in earlier_state.values["messages"]])


## 4. LangGraph — mémoire long terme (`Store`)

Le `Store` est indépendant des threads : on y écrit/lit des souvenirs
namespacés (typiquement par utilisateur), avec recherche sémantique si on lui
fournit une fonction d'embedding.


In [ ]:
from langgraph.store.memory import InMemoryStore


def embed_fn(texts):
    return [toy_embed(t).tolist() for t in texts]


store = InMemoryStore(index={"embed": embed_fn, "dims": 256})

user_id = "user-42"
namespace = (user_id, "memories")

store.put(namespace, str(uuid.uuid4()), {"fact": "Préfère des réponses concises, code d'abord"})
store.put(namespace, str(uuid.uuid4()), {"fact": "Équipe data engineering, utilise Airflow"})
store.put(namespace, str(uuid.uuid4()), {"fact": "N'aime pas les emojis dans les réponses"})

hits = store.search(namespace, query="comment dois-je formater mes réponses pour cet utilisateur ?")
for h in hits:
    print(f"  score={h.score:.3f}  {h.value}")


Chaque résultat porte un score de similarité — c'est la même mécanique
« cosinus » de la section 1, appliquée cette fois à de vrais souvenirs
persistés, scoping par `namespace` (ici par utilisateur).


In [ ]:
# Mémoire pilotée par l'agent lui-même (langmem) — nécessite un modèle tool-calling réel
if HAS_ANTHROPIC:
    from langmem import create_manage_memory_tool, create_search_memory_tool
    from langgraph.prebuilt import create_react_agent

    memory_agent = create_react_agent(
        "anthropic:claude-haiku-4-5-20251001",
        tools=[
            create_manage_memory_tool(namespace=("memories",)),
            create_search_memory_tool(namespace=("memories",)),
        ],
        store=store,
    )
    out = memory_agent.invoke({"messages": [
        HumanMessage(content="Retiens que je travaille sur des pipelines de données de santé, "
                              "donc sois particulièrement prudent sur la confidentialité.")
    ]})
    print(out["messages"][-1].content)
else:
    print("ANTHROPIC_API_KEY absente -> démo 'langmem' (agent auto-gérant sa mémoire) sautée.")
    print("Le mécanisme store.put/search ci-dessus reste, lui, entièrement fonctionnel hors-ligne.")


## 5. DeepAgents — sous-agents, filesystem virtuel, persistance cross-session

Cette section nécessite le package `deepagents` **et** un modèle tool-calling
réel — elle est donc protégée par un import et une vérification de clé API,
pour ne pas casser l'exécution du reste du notebook si l'un des deux manque.


In [ ]:
try:
    from deepagents import create_deep_agent
    from deepagents.backends import CompositeBackend, StateBackend, StoreBackend
    DEEPAGENTS_AVAILABLE = True
except ImportError:
    DEEPAGENTS_AVAILABLE = False
    print("Package 'deepagents' non installé -> `pip install deepagents` pour exécuter cette section.")

RUN_DEEPAGENTS_DEMO = DEEPAGENTS_AVAILABLE and HAS_ANTHROPIC
if DEEPAGENTS_AVAILABLE and not HAS_ANTHROPIC:
    print("deepagents est installé, mais ANTHROPIC_API_KEY est absente -> démo sautée.")


In [ ]:
if RUN_DEEPAGENTS_DEMO:
    cross_session_store = InMemoryStore()

    backend = lambda rt: CompositeBackend(
        default=StateBackend(rt),                          # brouillon éphémère, scope = thread
        routes={"/memories/": StoreBackend(rt)},            # tout sous /memories/ survit entre sessions
    )

    agent = create_deep_agent(
        model="anthropic:claude-sonnet-4-5",
        backend=backend,
        store=cross_session_store,
    )

    thread_a = {"configurable": {"thread_id": "deepagents-demo-A"}}
    agent.invoke(
        {"messages": [HumanMessage(content=(
            "Résume en 2 phrases la différence entre mémoire courte et longue terme "
            "dans LangGraph, et écris ce résumé dans /memories/notes.md"
        ))]},
        config=thread_a,
    )

    # Nouveau thread, MÊME store -> le fichier sous /memories/ doit être visible
    thread_b = {"configurable": {"thread_id": "deepagents-demo-B"}}
    result_b = agent.invoke(
        {"messages": [HumanMessage(content="Lis /memories/notes.md et cite-le.")]},
        config=thread_b,
    )
    print(result_b["messages"][-1].content)
else:
    print("Démo DeepAgents non exécutée (voir message ci-dessus). "
          "Relisez le pattern dans les slides (partie 2, backends) en attendant.")


**À observer en live si vous avez une clé API :** le thread B n'a jamais « vu »
la conversation du thread A — pourtant il retrouve le fichier, parce que
`/memories/` est routé vers un `StoreBackend` qui partage le même `store` que
le thread A. Un fichier écrit *hors* de `/memories/` (donc sur le
`StateBackend` par défaut) n'aurait, lui, pas survécu au changement de thread —
à tester en exercice.


## Pour aller plus loin

- Slides théorie : `docs/slides/agent-memory-part1-theory.html`
- Slides pratique : `docs/slides/agent-memory-part2-practice.html`
- CoALA — <https://arxiv.org/abs/2309.02427>
- Generative Agents — <https://arxiv.org/abs/2304.03442>
- MemGPT — <https://arxiv.org/abs/2310.08560>
- LangGraph — persistence & memory (docs.langchain.com/oss/python/langgraph)
- DeepAgents — <https://github.com/langchain-ai/deepagents>
- Anthropic, *Effective context engineering for AI agents* (anthropic.com/engineering)
- Chroma Research, *Context Rot* (2025) — <https://www.trychroma.com/research/context-rot>
